# Feature generation

## Imports and constants

In [1]:
import glob
import os
import time

import numpy as np
from scipy.special import airy, roots_legendre

NRM = 0.5 * np.log(np.pi / 2)
_G16, _W16 = roots_legendre(16)
_G24, _W24 = roots_legendre(24)
_TAU_S = 0.9

## Exact background functions (from `genesis_background.py`)

In [2]:
from genesis_background import z_squared, omega2, L0_logC

## Feature extraction and split cache

In [3]:
_FEATURE_CACHE_KEYS = ("uni", "zeta", "F1", "F2", "F3", "L0", "logC",
                       "lnR", "lam", "d", "a0", "k")


_CACHE_PART_MAX_BYTES = 90 * 1024**2

def feature_cache_base(scan, cache):
    if cache is False:
        return None
    if cache is None:
        return os.path.join(os.path.dirname(scan) or ".", "fit_curve_features")
    root, ext = os.path.splitext(cache)
    return root if ext.lower() == ".npz" else cache

def _part_index(path):
    try:
        return int(path.rsplit("_part", 1)[1].split(".npz")[0])
    except (IndexError, ValueError):
        return -1

def feature_cache_paths(cache_base):
    """Existing cache parts for this base, sorted by 1-based part index."""
    return sorted((p for p in glob.glob(f"{cache_base}_part*.npz")
                   if _part_index(p) >= 1), key=_part_index)

def load_split_feature_cache(cache_base):
    paths = feature_cache_paths(cache_base)
    if not paths:
        return None
    parts, count, seen = [], None, []
    for p in paths:
        with np.load(p) as z:
            missing = set(_FEATURE_CACHE_KEYS).difference(z.files)
            if missing:
                print(f"split feature cache missing {sorted(missing)}; rebuilding", flush=True)
                return None
            if "_part_count" in z.files:
                count = int(z["_part_count"]); seen.append(int(z["_part_index"]))
            parts.append({k: z[k] for k in _FEATURE_CACHE_KEYS})
    if count is not None and (len(paths) != count
                              or sorted(seen) != list(range(1, count + 1))):
        print(f"incomplete split cache ({len(paths)}/{count} parts); rebuilding",
              flush=True)
        return None
    print(f"features loaded from split cache -> {len(paths)} parts", flush=True)
    return {k: np.concatenate([part[k] for part in parts], axis=0)
            for k in _FEATURE_CACHE_KEYS}

def save_split_feature_cache(cache_base, out, max_bytes=_CACHE_PART_MAX_BYTES):
    n = len(out["d"])
    bytes_per_row = sum(np.asarray(out[k]).itemsize * (np.asarray(out[k]).size // n)
                        for k in _FEATURE_CACHE_KEYS)
    rows_per_chunk = max(1, int(max_bytes // bytes_per_row))
    n_parts = -(-n // rows_per_chunk)            # ceil division
    for old in feature_cache_paths(cache_base):  # drop stale parts from a prior run
        os.remove(old)
    for i, a in enumerate(range(0, n, rows_per_chunk), start=1):
        b = min(a + rows_per_chunk, n)
        path = f"{cache_base}_part{i}.npz"
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        np.savez_compressed(path, _part_index=i, _part_count=n_parts,
                            **{k: out[k][a:b] for k in _FEATURE_CACHE_KEYS})
        print(f"features split cache -> {path} rows {a}:{b}", flush=True)
    return n_parts

def build_features(scan, cache=None):
    cache_base = feature_cache_base(scan, cache)
    if cache_base is not None:
        cached = load_split_feature_cache(cache_base)
        if cached is not None:
            return cached
    S = np.load(scan)
    D, A0, K, X, RA = S["d"], S["alpha0"], S["kappa"], S["x"], S["R_abs"]
    n, m = X.shape
    out = {k: np.empty((n, m)) for k in ("uni", "zeta", "F1", "F2", "F3")}
    L0v = np.empty(n); LCv = np.empty(n)
    t0 = time.time()
    for i in range(n):
        d, a0, k = float(D[i]), float(A0[i]), float(K[i])
        x = X[i].astype(float); y = 1.0 + x
        z = np.sqrt(np.abs(z_squared(x, d, a0, k))); lnz = np.log(z)
        o2 = omega2(x, d, a0, k)
        xt = 1.0/k
        for _ in range(6):
            h = 1e-4; f = omega2(xt, d, a0, k)
            fp = (omega2(xt*(1+h), d, a0, k)
                  - omega2(xt*(1-h), d, a0, k))/(2*h*xt)
            xt = xt - np.clip(f/fp, -0.5*xt, 0.5*xt)
        s = (omega2(xt*(1+1e-4), d, a0, k)
             - omega2(xt*(1-1e-4), d, a0, k))/(2e-4*xt)
        yt = 1.0 + xt
        zeta = -np.sign(s)*np.abs(s)**(1/3)*(x - xt)
        zc = np.clip(zeta, -1e6, 40.0)
        Ai, _, Bi, _ = airy(zc); M2 = Ai**2 + Bi**2
        P = (np.abs(zc)/np.maximum(np.abs(o2), 1e-300))**0.25
        P = np.where(np.abs(x - xt) < x*1e-3, np.abs(s)**(-1/6.0), P)
        tA = np.minimum(np.sqrt(np.maximum(1.0 - y/yt, 0.0)), _TAU_S)
        tau = 0.5*tA[:, None]*(_G16[None, :] + 1.0); yij = yt*(1.0 - tau**2)
        PHI = ((0.5*tA)[:, None]
               * np.sqrt(np.maximum(-omega2(yij - 1.0, d, a0, k), 0.0))
               * 2.0*yt*tau*_W16[None, :]).sum(axis=1)
        ys = yt*(1.0 - _TAU_S**2); blk = y < ys
        if blk.any():
            lu0 = np.log(y[blk]); lu1 = np.log(ys)
            uij = (0.5*(lu1 - lu0)[:, None]*(_G24[None, :] + 1.0)
                   + lu0[:, None]); yb = np.exp(uij)
            PHI[blk] += ((0.5*(lu1 - lu0))[:, None]*yb
                         * np.sqrt(np.maximum(-omega2(yb - 1.0, d, a0, k),
                                              0.0))
                         * _W24[None, :]).sum(axis=1)
        lo2 = 0.5*np.log(np.maximum(np.abs(o2), 1e-300))
        out["uni"][i] = NRM + np.log(P) + 0.5*np.log(M2) - lnz
        out["zeta"][i] = zeta
        out["F1"][i] = PHI[0] - PHI
        out["F2"][i] = lo2 - lo2[0]
        out["F3"][i] = lnz - lnz[0]
        L0v[i], LCv[i] = L0_logC(d, a0, k)
        if i % 1024 == 0:
            print(f"  features {i}/{n}  [{time.time()-t0:.0f} s]", flush=True)
    out.update(L0=L0v, logC=LCv, lnR=np.log(RA), lam=np.log(K[:, None]*X),
               d=D, a0=A0, k=K)
    if cache_base is not None:
        n_parts = save_split_feature_cache(cache_base, out)
        print(f"features cached in {n_parts} npz files  [{time.time()-t0:.0f} s]")
    else:
        print(f"features computed in memory  [{time.time()-t0:.0f} s]")
    return out

## Configuration

In [4]:
DATA_DIR = "data"
SCAN = None              
FEATURE_CACHE = None       

## Build (or load) the feature cache

In [5]:
os.makedirs(DATA_DIR, exist_ok=True)
scan = SCAN
if scan is None:
    scan = os.path.join(DATA_DIR, "vcdm_R_scan.npz")
if not os.path.exists(scan):
    raise FileNotFoundError(
        f"Scan file not found: {scan} (run vcdm_R_evolution.ipynb first).")

cache_arg = FEATURE_CACHE
if cache_arg is None:
    cache_arg = os.path.join(DATA_DIR, "fit_curve_features")


F = build_features(scan, cache_arg)
n, m = F["lnR"].shape
print(f"\nfeatures ready: {n} curves x {m} points")
print("cached keys:", ", ".join(_FEATURE_CACHE_KEYS))
print("param ranges:  d in [%.2f, %.2f]   kappa in [%.1e, %.1e]"
      % (F["d"].min(), F["d"].max(), F["k"].min(), F["k"].max()))

  features 0/4096  [0 s]
  features 1024/4096  [16 s]
  features 2048/4096  [33 s]
  features 3072/4096  [49 s]
features split cache -> data\fit_curve_features_part1.npz rows 0:1122
features split cache -> data\fit_curve_features_part2.npz rows 1122:2244
features split cache -> data\fit_curve_features_part3.npz rows 2244:3366
features split cache -> data\fit_curve_features_part4.npz rows 3366:4096
features cached in 4 npz files  [83 s]

features ready: 4096 curves x 1500 points
cached keys: uni, zeta, F1, F2, F3, L0, logC, lnR, lam, d, a0, k
param ranges:  d in [0.10, 0.40]   kappa in [1.0e-09, 1.0e-04]
